# 06 — Stage-2 Ranker: LGBMRanker + Holdout Evaluation

**Purpose:** train the 'Judge'. Score the ~candidates per user precisely and put the best on top.

Built from empty. Key correctness points:
- `LGBMRanker(objective='lambdarank')`.
- **Sort rows by `customer_id` and pass `group` sizes** — the #1 silent bug in LightGBM ranking.
- Train on `ranking_train` (TRAIN week), evaluate MAP@12 on `ranking_val` (VAL holdout),
  compare to the popularity baseline from notebook 02, and inspect feature importance.

In [1]:
from recsys_utils import *
import numpy as np, pandas as pd, pickle
import lightgbm as lgb
mlflow = setup_mlflow()

train_df  = load_parquet(RANKING_DIR / 'ranking_train.parquet')
val_df    = load_parquet(RANKING_DIR / 'ranking_val.parquet')
val_truth = load_parquet(PROCESSED_DIR / 'val_truth.parquet')
print('train:', train_df.shape, ' val:', val_df.shape)

train: (1731315, 32)  val: (9280737, 32)


In [2]:
META_COLS = ['product_type_name', 'product_group_name', 'colour_group_name',
             'department_name', 'index_group_name', 'garment_group_name']
DROP = {'customer_id', 'article_id', 'label', 'source'}
feature_cols = [c for c in train_df.columns if c not in DROP]
cat_cols     = [c for c in META_COLS if c in feature_cols]

# LightGBM reads pandas 'category' dtype natively; make sure it is set on both frames.
for c in cat_cols:
    train_df[c] = train_df[c].astype('category')
    val_df[c]   = val_df[c].astype('category')
print('features:', len(feature_cols), ' categorical:', cat_cols)

features: 28  categorical: ['product_type_name', 'product_group_name', 'colour_group_name', 'department_name', 'index_group_name', 'garment_group_name']


In [3]:
# MUST sort by the group key and pass per-user group sizes (in the same order).
train_df = train_df.sort_values('customer_id').reset_index(drop=True)
group_sizes = train_df.groupby('customer_id', sort=True).size().values
assert group_sizes.sum() == len(train_df)

In [4]:
ranker = lgb.LGBMRanker(
    objective='lambdarank', metric='ndcg',
    n_estimators=400, learning_rate=0.05, num_leaves=63,
    reg_lambda=1.0, random_state=RANDOM_SEED, n_jobs=-1,
)
ranker.fit(
    train_df[feature_cols], train_df['label'],
    group=group_sizes, categorical_feature=cat_cols, eval_at=[TOP_K],
)
print('Ranker trained.')

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.018802 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4002
[LightGBM] [Info] Number of data points in the train set: 1731315, number of used features: 28
Ranker trained.


In [5]:
# Score the holdout and take top-12 per user.
val_df = val_df.sort_values('customer_id').reset_index(drop=True)
val_df['score'] = ranker.predict(val_df[feature_cols])

top = (val_df.sort_values(['customer_id', 'score'], ascending=[True, False])
             .groupby('customer_id')['article_id']
             .apply(lambda s: s.head(TOP_K).tolist()).to_dict())

users   = val_truth['customer_id'].tolist()
actuals = val_truth['actual'].tolist()
preds   = [top.get(u, []) for u in users]

ranker_map  = mapk(actuals, preds, TOP_K)
ranker_ndcg = mean_ndcg(actuals, preds, TOP_K)
print(f'RANKER   MAP@12 = {ranker_map:.5f}   NDCG@12 = {ranker_ndcg:.5f}')
print('Compare against the popularity baseline MAP@12 from notebook 02.')

RANKER   MAP@12 = 0.02804   NDCG@12 = 0.04174
Compare against the popularity baseline MAP@12 from notebook 02.


In [6]:
# Feature importance — what actually moved the needle.
imp = (pd.Series(ranker.feature_importances_, index=feature_cols)
         .sort_values(ascending=False))
print(imp.head(20))

department_name          2852
user_days_since_last     2237
user_avg_spend           2106
ui_price_ratio           2087
user_n_distinct          1696
user_n_tx                1474
colour_group_name        1319
item_n_tx                1285
user_cnt_90d             1206
product_type_name        1197
item_sales_7d            1093
als_score                1066
als_rank                  837
item_sales_prior7d        767
item_days_since_first     733
user_cnt_30d              722
item_velocity             702
item_avg_price            499
user_cnt_7d               390
trending_rank             359
dtype: int32


In [7]:
# Persist the trained ranker for the submission notebook.
with open(MODEL_DIR / 'lgbm_ranker.pkl', 'wb') as f:
    pickle.dump({'ranker': ranker, 'feature_cols': feature_cols, 'cat_cols': cat_cols}, f)

with mlflow.start_run(run_name='06_ranker_lgbm'):
    mlflow.log_params({'n_estimators': 400, 'learning_rate': 0.05,
                       'num_leaves': 63, 'reg_lambda': 1.0, 'n_features': len(feature_cols)})
    mlflow.log_metric('ranker_map_at_12', ranker_map)
    mlflow.log_metric('ranker_ndcg_at_12', ranker_ndcg)
print('Saved ranker + logged run.')

2026/06/11 13:05:48 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



Saved ranker + logged run.


## (Optional) CatBoost comparison — your action item

CatBoost handles categorical features natively. Flip the flag to run a head-to-head once
the LightGBM pipeline is solid. Keep whichever scores higher; this is optional, not required.

In [8]:
RUN_CATBOOST = False
if RUN_CATBOOST:
    from catboost import CatBoostRanker, Pool
    g_train = train_df['customer_id'].values
    pool = Pool(train_df[feature_cols], label=train_df['label'],
                group_id=g_train, cat_features=cat_cols)
    cb = CatBoostRanker(loss_function='YetiRank', iterations=400,
                        learning_rate=0.05, random_seed=RANDOM_SEED, verbose=False)
    cb.fit(pool)
    val_df['score_cb'] = cb.predict(val_df[feature_cols])
    top_cb = (val_df.sort_values(['customer_id', 'score_cb'], ascending=[True, False])
                    .groupby('customer_id')['article_id']
                    .apply(lambda s: s.head(TOP_K).tolist()).to_dict())
    cb_map = mapk(actuals, [top_cb.get(u, []) for u in users], TOP_K)
    print(f'CatBoost MAP@12 = {cb_map:.5f}  (LightGBM = {ranker_map:.5f})')